# 팬덤결속 지수(Fandom Cohesion Index) — 최종 코퍼스 10,020건 검증 노트북

원본 `data/v7_final/fandom_cohesion_index_v7.json`(10,020건, 5개 결속 활동 유형 A~E, D 정밀도 게이트)을 코퍼스와 대조해 재검증하고,
동결 스냅샷(7,350건, K=10/M=5)과 라이브(10,020건) 두 LDA 점수의 "결속형(팬클럽·기부·커뮤니티)" 비중과도 대조한다.

In [1]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 140)


def find_repo_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "data" / "v7_final" / "fandoms_v3_100.json").exists():
            return p
    raise FileNotFoundError("저장소 루트(data/v7_final/fandoms_v3_100.json)를 찾지 못함 — 저장소 안에서 실행하세요")


REPO = find_repo_root()
DATA_DIR = REPO / "data" / "v7_final"                       # 최종 산출물(10,020건 라이브 + 동결 스냅샷 7,350건)
ROUNDS_DIR = REPO / "data" / "v7_rounds"                    # 병합 로그 r1~r72


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def flatten_bullets(fandoms):
    rows = []
    for rec in fandoms:
        for kind in ("loyalty", "spillover"):
            for item in rec.get(kind, []):
                rows.append({"fandom": rec["fandom"], "category": rec.get("category"),
                             "bullet_type": kind, "text": item.get("t", "") or "", "url": item.get("u", "") or ""})
    return pd.DataFrame(rows)

coh = load_json(DATA_DIR / "fandom_cohesion_index_v7.json")
fandoms = load_json(DATA_DIR / "fandoms_v3_100.json")
frozen_scores = load_json(DATA_DIR / "fandom_scores_v6.json")                 # 동결 v7-40 스냅샷 7,350건, K=10/M=5
live_scores = load_json(DATA_DIR / "fandom_scores_live_reference_v7.json")    # 라이브 10,020건, K=8/M=5(게이트 미통과)
bullets_df = flatten_bullets(fandoms)
print("근거문장:", len(bullets_df), "| JSON total_bullets:", coh["total_bullets"])
print("결속 불릿:", coh["total_cohesion_bullets"], f"({coh['corpus_cohesion_share']:.1%})", "| 1건 이상 팬덤:", coh["n_fandoms_with_any_cohesion_bullet"])
print("유형:", coh["categories"])
print("D 게이트 정밀도 검증:", coh["d_gate_precision_check"])

근거문장: 10020 | JSON total_bullets: 10020
결속 불릿: 923 (9.2%) | 1건 이상 팬덤: 99
유형: ['A_공식팬클럽·회원제', 'B_팬카페·온라인커뮤니티', 'C_팬덤정체성·문화', 'E_오프라인결집·이벤트', 'D_기부·후원캠페인(팬덤주도)']
D 게이트 정밀도 검증: {'note': 'D 카테고리 게이트가 실제로 아티스트 개인 선행을 걸러내고 있는지의 재현 가능한 검증치.', '기부·후원류_전체_언급': 364, '게이트로_제외된_건수': 166, '제외_비율': 0.456}


## 1. 데이터 무결성 검증

In [2]:
per_fandom_corpus = bullets_df.groupby("fandom").size().to_dict()
rows = []; n_mis = share_mis = top_mis = 0
for rec in coh["fandoms"]:
    if per_fandom_corpus.get(rec["fandom"]) != rec["n_total_bullets"]: n_mis += 1
    if abs(rec["n_cohesion_bullets"] / rec["n_total_bullets"] - rec["cohesion_share"]) > 0.0005: share_mis += 1
    if rec["category_counts"] and rec["category_counts"].get(rec["top_category"], -1) != max(rec["category_counts"].values()): top_mis += 1  # 동점 허용
    row = {"팬덤": rec["fandom"], "구분": rec["category"], "근거문장수": rec["n_total_bullets"], "결속문장수": rec["n_cohesion_bullets"],
           "결속비중": rec["cohesion_share"], "대표유형": rec["top_category"]}
    row.update({c: rec["category_counts"].get(c, 0) for c in coh["categories"]})
    rows.append(row)
coh_df = pd.DataFrame(rows)
cat_sum = {c: int(coh_df[c].sum()) for c in coh["categories"]}
print(f"근거문장 수 != 코퍼스 실측: {n_mis} / {len(coh_df)} | cohesion_share 재계산 불일치: {share_mis} | top_category가 최다 유형(동점 포함)이 아님: {top_mis}")
print(f"결속문장수 합 {coh_df['결속문장수'].sum()} == total_cohesion_bullets {coh['total_cohesion_bullets']} :", coh_df["결속문장수"].sum() == coh["total_cohesion_bullets"])
print("유형별 합 == category_totals :", cat_sum == coh["category_totals"], cat_sum)
print(f"1건 이상 팬덤: {(coh_df['결속문장수'] > 0).sum()} (JSON {coh['n_fandoms_with_any_cohesion_bullet']})")

근거문장 수 != 코퍼스 실측: 0 / 100 | cohesion_share 재계산 불일치: 0 | top_category가 최다 유형(동점 포함)이 아님: 0
결속문장수 합 923 == total_cohesion_bullets 923 : True
유형별 합 == category_totals : True {'A_공식팬클럽·회원제': 480, 'B_팬카페·온라인커뮤니티': 190, 'C_팬덤정체성·문화': 122, 'E_오프라인결집·이벤트': 173, 'D_기부·후원캠페인(팬덤주도)': 198}
1건 이상 팬덤: 99 (JSON 99)


## 2. 팬덤별 결속 문장 수·비중 — 상위 15

In [3]:
coh_df = coh_df.sort_values(["결속문장수", "결속비중"], ascending=[False, False]).reset_index(drop=True)
coh_df.index = coh_df.index + 1
coh_df.head(15)

,팬덤,구분,근거문장수,결속문장수,결속비중,대표유형,A_공식팬클럽·회원제,B_팬카페·온라인커뮤니티,C_팬덤정체성·문화,E_오프라인결집·이벤트,D_기부·후원캠페인(팬덤주도)
1,임영웅,트로트,131,30,0.2290,A_공식팬클럽·회원제,18,13,2,1,11
2,김재중,솔로,77,25,0.3247,E_오프라인결집·이벤트,9,4,2,11,1
3,박서진,트로트,99,25,0.2525,A_공식팬클럽·회원제,18,7,2,1,12
4,정동원,트로트,107,24,0.2243,A_공식팬클럽·회원제,19,5,0,0,15
5,김호중,트로트,91,22,0.2418,B_팬카페·온라인커뮤니티,9,13,2,0,8
6,이찬원,트로트,106,20,0.1887,B_팬카페·온라인커뮤니티,7,10,2,0,9
7,잭스키스,원로그룹,85,18,0.2118,A_공식팬클럽·회원제,15,1,3,2,4
8,리센느(RESCENE),K-pop 걸그룹,85,18,0.2118,D_기부·후원캠페인(팬덤주도),2,3,3,4,8
9,송가인,트로트,92,17,0.1848,A_공식팬클럽·회원제,10,8,2,1,7
10,god,원로그룹,99,17,0.1717,A_공식팬클럽·회원제,12,0,2,4,2


## 3. 강조 3팬덤 — BTS·임영웅·리센느(RESCENE) (KEY_FINDINGS: 15건 6.6% / 30건 23% 전체 1위 / 18건 21.2%)

In [4]:
for name in ["BTS", "임영웅", "리센느(RESCENE)"]:
    r = coh_df[coh_df["팬덤"] == name]
    print(f"{name}: 결속 {int(r['결속문장수'].iloc[0])}건 / {int(r['근거문장수'].iloc[0])}건 = {r['결속비중'].iloc[0]:.1%} "
          f"(결속문장수 {r.index[0]}위, 대표유형 {r['대표유형'].iloc[0]})")
rank_by_share = coh_df.sort_values("결속비중", ascending=False).reset_index(drop=True)
print("결속비중 1위:", rank_by_share.iloc[0]["팬덤"], f"{rank_by_share.iloc[0]['결속비중']:.1%}")

BTS: 결속 15건 / 227건 = 6.6% (결속문장수 15위, 대표유형 D_기부·후원캠페인(팬덤주도))
임영웅: 결속 30건 / 131건 = 22.9% (결속문장수 1위, 대표유형 A_공식팬클럽·회원제)
리센느(RESCENE): 결속 18건 / 85건 = 21.2% (결속문장수 8위, 대표유형 D_기부·후원캠페인(팬덤주도))
결속비중 1위: 김재중 32.5%


## 4. 유형별(A~E) 분포와 카테고리별 평균

In [5]:
type_df = pd.DataFrame([{"유형": c, "건수": coh["category_totals"][c]} for c in coh["categories"]]).sort_values("건수", ascending=False)
type_df["비중(멀티라벨 합 대비)"] = (type_df["건수"] / type_df["건수"].sum()).round(4)
print(type_df.to_string(index=False))
print()
cat_df = (coh_df.groupby("구분")["결속비중"].agg(["mean", "count"]).rename(columns={"mean": "평균 결속비중", "count": "팬덤 수"})
          .sort_values("평균 결속비중", ascending=False))
cat_df["평균 결속비중"] = cat_df["평균 결속비중"].round(4)
cat_df

              유형  건수  비중(멀티라벨 합 대비)
     A_공식팬클럽·회원제 480         0.4127
D_기부·후원캠페인(팬덤주도) 198         0.1702
   B_팬카페·온라인커뮤니티 190         0.1634
    E_오프라인결집·이벤트 173         0.1488
      C_팬덤정체성·문화 122         0.1049



,평균 결속비중,팬덤 수
구분,,
원로그룹,0.1691,3
트로트,0.1652,10
솔로,0.1075,17
K-pop 보이그룹 /록,0.0856,3
K-pop 보이그룹,0.0830,18
혼성,0.0811,1
K-pop 걸그룹,0.0807,23
발라드,0.0735,16
힙합,0.0585,7


## 5. LDA 결속형 비중과의 대조 — 동결 스냅샷(보고서 본문)과 라이브 재적합 모두

In [6]:
def check_factor_share(scores, label):
    sum_mismatch, dominant_mismatch = [], []
    for d in scores:
        shares = d["factor_share"]
        if abs(sum(shares.values()) - 1.0) > 0.001:
            sum_mismatch.append((d["fandom"], round(sum(shares.values()), 4)))
        if max(shares, key=shares.get) != d["dominant_factor"]:
            dominant_mismatch.append((d["fandom"], max(shares, key=shares.get), d["dominant_factor"]))
    print(f"[{label}] factor_share 합 != 1.0 인 팬덤 수: {len(sum_mismatch)} / {len(scores)}")
    print(f"[{label}] dominant_factor 재계산 불일치 팬덤 수: {len(dominant_mismatch)} / {len(scores)}")
    return sum_mismatch, dominant_mismatch

check_factor_share(frozen_scores, "동결 7,350건 K=10/M=5")
check_factor_share(live_scores, "라이브 10,020건 K=8/M=5")
FACTOR = "결속형(팬클럽·기부·커뮤니티)"
coh_df["LDA 결속형(동결)"] = coh_df["팬덤"].map({d["fandom"]: d["factor_share"].get(FACTOR, 0.0) for d in frozen_scores})
coh_df["LDA 결속형(라이브)"] = coh_df["팬덤"].map({d["fandom"]: d["factor_share"].get(FACTOR, 0.0) for d in live_scores})
for col in ["LDA 결속형(동결)", "LDA 결속형(라이브)"]:
    sub = coh_df.dropna(subset=[col])
    r_p = np.corrcoef(sub["결속비중"], sub[col])[0, 1]
    r_s = sub[["결속비중", col]].corr(method="spearman").iloc[0, 1]
    print(f"{col}: n={len(sub)}, Pearson r={r_p:.4f}, Spearman rho={r_s:.4f}")
print("동결 스냅샷 로스터에 없는 팬덤(라이브 전용):", sorted(set(coh_df["팬덤"]) - {d["fandom"] for d in frozen_scores}))
print("동결에서 dominant_factor가 결속형인 팬덤:", [d["fandom"] for d in frozen_scores if d["dominant_factor"] == FACTOR])
coh_df[["팬덤", "결속비중", "LDA 결속형(동결)", "LDA 결속형(라이브)"]].head(10)

[동결 7,350건 K=10/M=5] factor_share 합 != 1.0 인 팬덤 수: 0 / 100
[동결 7,350건 K=10/M=5] dominant_factor 재계산 불일치 팬덤 수: 0 / 100
[라이브 10,020건 K=8/M=5] factor_share 합 != 1.0 인 팬덤 수: 0 / 100
[라이브 10,020건 K=8/M=5] dominant_factor 재계산 불일치 팬덤 수: 0 / 100
LDA 결속형(동결): n=97, Pearson r=0.6867, Spearman rho=0.6043
LDA 결속형(라이브): n=100, Pearson r=0.2736, Spearman rho=0.0786
동결 스냅샷 로스터에 없는 팬덤(라이브 전용): ['몬스타엑스', '빈지노', '투어스(TWS)']
동결에서 dominant_factor가 결속형인 팬덤: ['김호중']


,팬덤,결속비중,LDA 결속형(동결),LDA 결속형(라이브)
1,임영웅,0.2290,0.2017,0.1013
2,김재중,0.3247,0.1341,0.1072
3,박서진,0.2525,0.2653,0.3180
4,정동원,0.2243,0.1997,0.1883
5,김호중,0.2418,0.2413,0.1976
6,이찬원,0.1887,0.2291,0.1242
7,잭스키스,0.2118,0.2167,0.0825
8,리센느(RESCENE),0.2118,0.1050,0.0760
9,송가인,0.1848,0.1456,0.0749
10,god,0.1717,0.1719,0.0667


## 6. 한계

1. A~E 유형별 키워드 목록과 D 게이트 공존 키워드(팬클럽/팬카페/팬덤/팬들/함께/자발적/정회원/회원)는 methodology 문구로만 남아 있고 키워드 전체 사전은 JSON에 없어 키워드 재매칭은 수행하지 않았다.
2. 결속 지수는 키워드 매칭 지표이고 LDA 결속형 비중은 토픽 혼합비중이라 계산 방식이 다르다 — 5절 상관은 "같은 방향을 가리키는가"만 본다.
3. 동결 스냅샷 로스터(한로로·pH-1·BE'O 포함)와 라이브 로스터(몬스타엑스·투어스(TWS)·빈지노 포함)는 3개 팬덤이 다르다(README 로스터 절).